In [2]:
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, classification_report

In [3]:
#import files

gt = pd.read_csv("/Users/saharshbhave/Downloads/Ground_Truth.csv")
pred = pd.read_csv("/Users/saharshbhave/Downloads/FS_GPT_Pred.csv")

#Column names
causality_col = "Causality"
relation_col = "Relation_Type"

In [4]:
#Relation Labels

Relation_Labels = [
    "Mechanistic/Theory",
    "Measurement/Method",
    "Description",
    "Correlation",
    "Hypothesis",
    "Association"
]

In [6]:
#Validate

if len(gt) != len(pred):
    raise ValueError(f"Row mismatch: gt{len(gt)}, pred{len(pred)}")

#Normalize values
def normalize_causality(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    if x in ["Yes", "yes"]:
        return "yes"
    if x in ["No", "no"]:
        return "no"
    
def normalize_relation(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


gt[causality_col] = gt[causality_col].apply(normalize_causality)
pred[causality_col] = pred[causality_col].apply(normalize_causality)

gt[relation_col] = gt[relation_col].apply(normalize_relation)
pred[relation_col] = pred[relation_col].apply(normalize_relation)

In [7]:
#Causality Metrics

y_true_c = gt[causality_col]
y_pred_c = pred[causality_col]

causality_labels = ["yes", "no"]

cm_c = confusion_matrix(y_true_c, y_pred_c, labels=causality_labels)
print(confusion_matrix)
print(pd.DataFrame(cm_c, index=causality_labels, columns=causality_labels))

f1_c = f1_score(y_true_c, y_pred_c, pos_label="yes", average="binary")
weighted_f1_c = f1_score(y_true_c, y_pred_c, average="weighted")

print(f"\nF1 Score: {f1_c:.4f}")
print(f"Weighted F1 Score: {weighted_f1_c:.4f}")

print("\nClassification Report:")
print(classification_report(y_true_c, y_pred_c, labels=causality_labels, zero_division=0))

<function confusion_matrix at 0x1181256c0>
     yes  no
yes   31   3
no     6  20

F1 Score: 0.8732
Weighted F1 Score: 0.8486

Classification Report:
              precision    recall  f1-score   support

         yes       0.84      0.91      0.87        34
          no       0.87      0.77      0.82        26

    accuracy                           0.85        60
   macro avg       0.85      0.84      0.84        60
weighted avg       0.85      0.85      0.85        60



In [9]:
# Relation Type Metrics
#only where causality = no

mask_no = gt[causality_col].astype(str).str.strip().str.lower() == "no"

y_true_r = gt.loc[mask_no, relation_col].copy()
y_pred_r = pred.loc[mask_no, relation_col].copy()

# normalize
y_true_r = y_true_r.fillna("").astype(str).str.strip().str.lower()
y_pred_r = y_pred_r.fillna("").astype(str).str.strip().str.lower()

print("Number of GT 'no' rows:", len(y_true_r))
print("Unique y_true relation values:", y_true_r.unique())
print("Unique y_pred relation values:", y_pred_r.unique())

# replace blank strings with placeholder
y_true_r = y_true_r.replace("", "missing_prediction")
y_pred_r = y_pred_r.replace("", "missing_prediction")

# build labels from actual data instead of fixed list
relation_labels = sorted(set(y_true_r) | set(y_pred_r))

print("Labels used for scoring:", relation_labels)

if len(y_true_r) == 0:
    print("No rows found where ground truth causality = 'no'. Relation metrics cannot be computed.")
else:
    cm_r = confusion_matrix(y_true_r, y_pred_r, labels=relation_labels)
    print("Confusion Matrix:")
    print(pd.DataFrame(cm_r, index=relation_labels, columns=relation_labels))

    f1_macro_r = f1_score(y_true_r, y_pred_r, average="macro")
    f1_weighted_r = f1_score(y_true_r, y_pred_r, average="weighted")

    print(f"\nF1 Score (Macro): {f1_macro_r:.4f}")
    print(f"Weighted F1 Score: {f1_weighted_r:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true_r, y_pred_r, labels=relation_labels, zero_division=0))

Number of GT 'no' rows: 26
Unique y_true relation values: ['mechanistic / theory' 'measurement/method' 'description' 'hypothesis']
Unique y_pred relation values: ['description' '' 'measurement/method' 'mechanistic/theory']
Labels used for scoring: ['description', 'hypothesis', 'measurement/method', 'mechanistic / theory', 'mechanistic/theory', 'missing_prediction']
Confusion Matrix:
                      description  hypothesis  measurement/method  \
description                     0           0                   2   
hypothesis                      0           0                   1   
measurement/method              0           0                   8   
mechanistic / theory            3           0                   5   
mechanistic/theory              0           0                   0   
missing_prediction              0           0                   0   

                      mechanistic / theory  mechanistic/theory  \
description                              0                   0  